# COX-2 model training

Run the cells top to bottom. Only **cell 1** needs editing.

**What happens:** the raw data is cleaned and featurised, the model is tuned and
trained twice (random split and scaffold split), it is checked against a
similarity baseline, and the final model is saved.

**Runtime:** roughly 30-50 minutes with the default settings. The GPU is not
used; scikit-learn random forests run on CPU. More cores means faster.

**Needs:** `rdkit scikit-learn optuna joblib pandas numpy matplotlib scipy`

## 1. Settings — edit this cell

In [ ]:
# Column names in the raw CSV (see cell 2 to print them)
SMILES_COL = "Smiles"
LABEL_COL = "Activity"

# Extra descriptor columns from the raw file to use as features.
# None  -> use every numeric column that is not SMILES or the label
# []    -> use only the features computed from SMILES
# [...] -> use exactly these
KEEP_COLS = None

# How long the hyperparameter search may run, PER split mode.
# Stops at whichever limit is reached first. "both" runs it twice.
N_TRIALS = 30
TIMEOUT_MIN = 45

# "both" = random + scaffold (recommended: the gap is a result in itself)
SPLIT_MODE = "both"

## 2. Load the raw data and check the columns

In [ ]:
import ethnohack.cox_model_v3.cox_preprocess as prep
import ethnohack.cox_model_v3.cox_train as train

raw = prep.load_raw()
print(f"{len(raw)} rows, {len(raw.columns)} columns\n")
print(raw.columns.tolist())
raw.head(3)

c:\Users\chiar\Desktop\Hackathon\ethnohack\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


4520 rows, 4 columns

['Molecule ChEMBL ID', 'Smiles', 'Activity', 'scaffold']


,Molecule ChEMBL ID,Smiles,Activity,scaffold
0,CHEMBL501208,CS(=O)(=O)c1ccc(-c2[nH]c3ccccc3c2-c2ccc(F)cc2)cc1,1,c1ccc(-c2[nH]c3ccccc3c2-c2ccccc2)cc1
1,CHEMBL499069,COc1ccc(-c2c(-c3ccc(S(C)(=O)=O)cc3)[nH]c3ccccc...,1,c1ccc(-c2[nH]c3ccccc3c2-c2ccccc2)cc1
2,CHEMBL525247,Cc1ccc2[nH]c(-c3ccc(S(N)(=O)=O)cc3)c(-c3ccccc3...,1,c1ccc(-c2[nH]c3ccccc3c2-c2ccccc2)cc1


**Stop here if** `SMILES_COL` or `LABEL_COL` are not in that list. Fix cell 1 first.

## 3. Preprocessing

Cleans the data, computes Murcko scaffolds (the ring core of each molecule,
used to group analogues), builds the feature table and writes it to disk.

In [ ]:
keep = KEEP_COLS
if keep is None:
    keep = [c for c in raw.columns if c not in (SMILES_COL, LABEL_COL)]

data = prep.build_dataset(
    smiles_col=SMILES_COL,
    label_col=LABEL_COL,
    keep_cols=keep,
)

cols = prep.feature_columns(data)
extra = [c for c in cols if not c.startswith("fp_")]
print(f"\n{len(cols)} feature columns, of which {len(extra)} are not fingerprint bits")
print(extra[:30])

Unparsable SMILES removed: 0
Stereo conflicts: 19 structures (55 rows) removed
Stereo duplicates: 64 rows removed
Compounds: 4401 | actives: 2416 (54.9%)
Unique scaffolds: 1314
Feature columns: 2048  (mode: ecfp)
Saved: C:\Users\chiar\Desktop\Hackathon\ethnohack\datasets\cox2_prepared.csv

2048 feature columns, of which 0 are not fingerprint bits
[]


### Can these features be computed for new molecules?

Important for later: the plant compounds are not in the raw file, so every
feature must be computable from a SMILES string alone. Columns listed as
missing below exist only in the training table and will break prediction.

In [ ]:
probe = prep.compute_rdkit_features(["CC(=O)Oc1ccccc1C(=O)O"])  # aspirin
missing = sorted(set(cols) - set(probe.columns))
print("Cannot be computed for new molecules:", missing or "none")

## 4. Train

**Random split:** close analogues land in both train and test, so the model can
recognise the compound series instead of the chemistry. Scores look good.

**Scaffold split:** molecules sharing a ring core stay on one side. Closer to
the real task (unseen plant metabolites). Lower, but honest. **Report this one.**

This is the long cell.

In [ ]:
results = train.run_models(
    data, SPLIT_MODE, n_trials=N_TRIALS, timeout_min=TIMEOUT_MIN
)
best = results[-1]  # scaffold run when SPLIT_MODE == "both"

In [ ]:
train.plot_roc(results)

## 5. Sanity checks

These decide what you can claim about the model. Note the numbers down.

### 5a. Is the model better than a similarity search?

Baseline: score each test compound by how similar it is to the nearest known
active. If the model barely beats this, it mostly learned "looks like a known
active" rather than anything about the chemistry. Worth reporting either way.

In [ ]:
sim = train.similarity_baseline(best)
combo = train.combined_baseline(best, sim)

### 5b. Where does the model actually work?

Test AUC split by structural similarity to the training set. The lowest bin
predicts how the model will behave on unfamiliar plant metabolites, which is
the number that matters for the project.

In [ ]:
bins = train.reliability_bins(best)

## 6. Save

Saves the model together with the feature settings and column order, plus the
training SMILES (needed for the reliability flag at prediction time).

`tune_cutoff=True` picks the 0/1 threshold by cross-validation on the training
data instead of defaulting to 0.5. The test set stays untouched.

In [ ]:
train.save_model(best, tune_cutoff=True)

## 7. Check the saved model loads

In [ ]:
import ethnohack.cox_model_v3.cox_model_api_v3 as cox

cox.reset_cache()
cox.model_info()

cox.predict(["CC(=O)Oc1ccccc1C(=O)O", "CC(C)Cc1ccc(cc1)C(C)C(=O)O"])  # aspirin, ibuprofen

---

## Reading the results

**Compare PR-AUC against the prevalence**, not against 0. Prevalence is the
share of actives, which is what random guessing already achieves.

**The probabilities are a ranking, not a verdict.** Use them to compare and
sort compounds, not to declare one active.

**Filter to `in_domain == True`** before drawing conclusions or running SHAP.
Predictions on molecules far from the training data are much less reliable.

## For the 5-LOX model

Point the constants at the 5-LOX paths and rerun this notebook unchanged.

## For TreeSHAP

```python
import shap
import ethnohack.cox_model.cox_model_api as cox

X = cox.featurize(smiles_list)
sv = shap.TreeExplainer(cox.load_model()).shap_values(X)
sv_act = sv[1] if isinstance(sv, list) else sv[..., 1]
shap.summary_plot(sv_act, X, feature_names=cox.feature_names())
```

Start with a few hundred molecules; TreeSHAP on a large forest with ~2000
features is slow. `cox.readable_names(smiles_list)` adds substructure hints to
the fingerprint column names for the final figure.